In [6]:
# 재방문·재구매 분석에 필요한 라이브러리와 데이터 기준 설정

from pathlib import Path
import duckdb

parquet_path = r"../data/processed/20*.parquet"

analysis_start_date = "2019-12-01"
analysis_end_date = "2020-05-01"

anomaly_dates = [
    "2020-01-01",
    "2020-01-02",
    "2020-01-03",
    "2020-02-27",
    "2020-04-20",
    "2020-04-21"
]

anomaly_dates_sql = ", ".join(
    f"DATE '{d}'" for d in anomaly_dates
)

In [3]:
# DuckDB 대용량 처리용 설정

temp_dir = Path(r"D:\duckdb_temp")
temp_dir.mkdir(exist_ok=True)

duckdb.sql("SET temp_directory = 'D:/duckdb_temp'")
duckdb.sql("SET preserve_insertion_order = false")
duckdb.sql("SET threads = 2")
duckdb.sql("SET memory_limit = '6GB'")

In [7]:
# 월별 활동 사용자 수 확인

duckdb.sql(f"""
    SELECT
        STRFTIME(event_time, '%Y-%m') AS month,
        COUNT(DISTINCT user_id) AS active_users

    FROM read_parquet('{parquet_path}')

    WHERE event_time >= '{analysis_start_date}'
      AND event_time < '{analysis_end_date}'

    GROUP BY month
    ORDER BY month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────────┐
│  month  │ active_users │
│ varchar │    int64     │
├─────────┼──────────────┤
│ 2019-12 │      4577232 │
│ 2020-01 │      4385985 │
│ 2020-02 │      4233206 │
│ 2020-03 │      4114060 │
│ 2020-04 │      4509623 │
└─────────┴──────────────┘



In [14]:
# 기준 월 활동 사용자 중 다음 달에도 활동한 사용자와 재방문율 계산
# 기준 월과 다음 달을 함께 표시

duckdb.sql(f"""
    WITH monthly_users AS (
        SELECT DISTINCT
            DATE_TRUNC('month', event_time) AS month,
            user_id

        FROM read_parquet('{parquet_path}')

        WHERE event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
    ),

    revisit AS (
        SELECT
            a.month,
            COUNT(DISTINCT a.user_id) AS base_users,
            COUNT(DISTINCT b.user_id) AS revisit_users

        FROM monthly_users a

        LEFT JOIN monthly_users b
            ON a.user_id = b.user_id
           AND b.month = a.month + INTERVAL '1 month'

        WHERE a.month < DATE '2020-04-01'

        GROUP BY a.month
    )

    SELECT
        STRFTIME(month, '%Y-%m') AS base_month,
        STRFTIME(month + INTERVAL '1 month', '%Y-%m') AS next_month,

        base_users,
        revisit_users,

        ROUND(
            revisit_users * 100.0 / base_users,
            2
        ) AS revisit_rate

    FROM revisit

    ORDER BY month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┬───────────────┬──────────────┐
│ base_month │ next_month │ base_users │ revisit_users │ revisit_rate │
│  varchar   │  varchar   │   int64    │     int64     │    double    │
├────────────┼────────────┼────────────┼───────────────┼──────────────┤
│ 2019-12    │ 2020-01    │    4577232 │       1797428 │        39.27 │
│ 2020-01    │ 2020-02    │    4385985 │       1702723 │        38.82 │
│ 2020-02    │ 2020-03    │    4233206 │       1659585 │         39.2 │
│ 2020-03    │ 2020-04    │    4114060 │       1526245 │         37.1 │
└────────────┴────────────┴────────────┴───────────────┴──────────────┘



In [17]:
# 월별 구매 사용자 중 다음 달에도 구매한 사용자와 재구매율 계산
# 1월 초와 4월 20~21일에 Purchase 로그 이상이 있었기 때문에 재구매율은 재방문율보다 데이터 품질 제한이 더 클 것으로 예상
# 기준 월과 다음 달을 함께 표시

duckdb.sql(f"""
    WITH monthly_buyers AS (
        SELECT DISTINCT
            DATE_TRUNC('month', event_time) AS month,
            user_id

        FROM read_parquet('{parquet_path}')

        WHERE event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND event_type = 'purchase'
    ),

    repurchase AS (
        SELECT
            a.month,
            COUNT(DISTINCT a.user_id) AS base_buyers,
            COUNT(DISTINCT b.user_id) AS repurchase_users

        FROM monthly_buyers a

        LEFT JOIN monthly_buyers b
            ON a.user_id = b.user_id
           AND b.month = a.month + INTERVAL '1 month'

        WHERE a.month < DATE '2020-04-01'

        GROUP BY a.month
    )

    SELECT
        STRFTIME(month, '%Y-%m') AS base_month,
        STRFTIME(month + INTERVAL '1 month', '%Y-%m') AS next_month,

        base_buyers,
        repurchase_users,

        ROUND(
            repurchase_users * 100.0 / base_buyers,
            2
        ) AS repurchase_rate

    FROM repurchase

    ORDER BY month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬─────────────┬──────────────────┬─────────────────┐
│ base_month │ next_month │ base_buyers │ repurchase_users │ repurchase_rate │
│  varchar   │  varchar   │    int64    │      int64       │     double      │
├────────────┼────────────┼─────────────┼──────────────────┼─────────────────┤
│ 2019-12    │ 2020-01    │      500997 │           102009 │           20.36 │
│ 2020-01    │ 2020-02    │      359105 │            93209 │           25.96 │
│ 2020-02    │ 2020-03    │      392356 │           104669 │           26.68 │
│ 2020-03    │ 2020-04    │      453487 │            88244 │           19.46 │
└────────────┴────────────┴─────────────┴──────────────────┴─────────────────┘



In [20]:
# 월→다음 달 재방문율과 재구매율을 한 표에서 비교

duckdb.sql(f"""
    WITH monthly_users AS (
        SELECT DISTINCT
            DATE_TRUNC('month', event_time) AS month,
            user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
    ),

    monthly_buyers AS (
        SELECT DISTINCT
            DATE_TRUNC('month', event_time) AS month,
            user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND event_type = 'purchase'
    ),

    revisit AS (
        SELECT
            a.month,
            COUNT(DISTINCT a.user_id) AS base_users,
            COUNT(DISTINCT b.user_id) AS revisit_users
        FROM monthly_users a
        LEFT JOIN monthly_users b
            ON a.user_id = b.user_id
           AND b.month = a.month + INTERVAL '1 month'
        WHERE a.month < DATE '2020-04-01'
        GROUP BY a.month
    ),

    repurchase AS (
        SELECT
            a.month,
            COUNT(DISTINCT a.user_id) AS base_buyers,
            COUNT(DISTINCT b.user_id) AS repurchase_users
        FROM monthly_buyers a
        LEFT JOIN monthly_buyers b
            ON a.user_id = b.user_id
           AND b.month = a.month + INTERVAL '1 month'
        WHERE a.month < DATE '2020-04-01'
        GROUP BY a.month
    )

    SELECT
        STRFTIME(r.month, '%Y-%m') AS base_month,
        STRFTIME(r.month + INTERVAL '1 month', '%Y-%m') AS next_month,

        r.base_users,
        r.revisit_users,
        ROUND(
            r.revisit_users * 100.0 / r.base_users,
            2
        ) AS revisit_rate,

        p.base_buyers,
        p.repurchase_users,
        ROUND(
            p.repurchase_users * 100.0 / p.base_buyers,
            2
        ) AS repurchase_rate

    FROM revisit r

    JOIN repurchase p
        ON r.month = p.month

    ORDER BY r.month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┬───────────────┬──────────────┬─────────────┬──────────────────┬─────────────────┐
│ base_month │ next_month │ base_users │ revisit_users │ revisit_rate │ base_buyers │ repurchase_users │ repurchase_rate │
│  varchar   │  varchar   │   int64    │     int64     │    double    │    int64    │      int64       │     double      │
├────────────┼────────────┼────────────┼───────────────┼──────────────┼─────────────┼──────────────────┼─────────────────┤
│ 2019-12    │ 2020-01    │    4577232 │       1797428 │        39.27 │      500997 │           102009 │           20.36 │
│ 2020-01    │ 2020-02    │    4385985 │       1702723 │        38.82 │      359105 │            93209 │           25.96 │
│ 2020-02    │ 2020-03    │    4233206 │       1659585 │         39.2 │      392356 │           104669 │           26.68 │
│ 2020-03    │ 2020-04    │    4114060 │       1526245 │         37.1 │      453487 │            88244 │           19.46 │
└────────────┴──

In [24]:
# 구매 경험이 있는 사용자는 구매하지 않은 사용자보다 다음 달에 다시 활동할 가능성이 높은지를 확인
# 구매 경험 여부에 따른 다음 달 재방문율 비교

duckdb.sql(f"""
    WITH monthly_user_behavior AS (
        SELECT
            DATE_TRUNC('month', event_time) AS month,
            user_id,

            MAX(
                CASE
                    WHEN event_type = 'purchase' THEN 1
                    ELSE 0
                END
            ) AS purchased

        FROM read_parquet('{parquet_path}')

        WHERE event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'

        GROUP BY
            month,
            user_id
    ),

    user_revisit AS (
        SELECT
            a.month,
            a.user_id,
            a.purchased,

            CASE
                WHEN b.user_id IS NOT NULL THEN 1
                ELSE 0
            END AS revisited

        FROM monthly_user_behavior a

        LEFT JOIN monthly_user_behavior b
            ON a.user_id = b.user_id
           AND b.month = a.month + INTERVAL '1 month'

        WHERE a.month < DATE '2020-04-01'
    )

    SELECT
        STRFTIME(month, '%Y-%m') AS base_month,
        STRFTIME(month + INTERVAL '1 month', '%Y-%m') AS next_month,

        CASE
            WHEN purchased = 1 THEN '구매 사용자'
            ELSE '비구매 사용자'
        END AS user_type,

        COUNT(*) AS user_count,
        SUM(revisited) AS revisit_users,

        ROUND(
            SUM(revisited) * 100.0 / COUNT(*),
            2
        ) AS revisit_rate

    FROM user_revisit

    GROUP BY
        month,
        purchased

    ORDER BY
        month,
        purchased DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬───────────────┬────────────┬───────────────┬──────────────┐
│ base_month │ next_month │   user_type   │ user_count │ revisit_users │ revisit_rate │
│  varchar   │  varchar   │    varchar    │   int64    │    int128     │    double    │
├────────────┼────────────┼───────────────┼────────────┼───────────────┼──────────────┤
│ 2019-12    │ 2020-01    │ 구매 사용자   │     500997 │        289998 │        57.88 │
│ 2019-12    │ 2020-01    │ 비구매 사용자 │    4076235 │       1507430 │        36.98 │
│ 2020-01    │ 2020-02    │ 구매 사용자   │     359105 │        205160 │        57.13 │
│ 2020-01    │ 2020-02    │ 비구매 사용자 │    4026880 │       1497563 │        37.19 │
│ 2020-02    │ 2020-03    │ 구매 사용자   │     392356 │        220412 │        56.18 │
│ 2020-02    │ 2020-03    │ 비구매 사용자 │    3840850 │       1439173 │        37.47 │
│ 2020-03    │ 2020-04    │ 구매 사용자   │     453487 │        247843 │        54.65 │
│ 2020-03    │ 2020-04    │ 비구매 사용자 │    3660573 │       1278402 │    

In [26]:
# [분석용 코드]
# 당월 사용자의 최대 퍼널 도달 단계별 다음 달 재방문율 비교

# 분류 기준 
#- View only
#   → 해당 월에 Cart/Purchase가 없는 사용자

#- Cart but no Purchase
#   → Cart는 했지만 Purchase는 없는 사용자

#- Purchase
#   → 해당 월에 Purchase가 1번이라도 있는 사용자

duckdb.sql(f"""
    WITH monthly_user_behavior AS (
        SELECT
            DATE_TRUNC('month', event_time) AS month,
            user_id,

            MAX(
                CASE WHEN event_type = 'cart'
                     THEN 1 ELSE 0 END
            ) AS had_cart,

            MAX(
                CASE WHEN event_type = 'purchase'
                     THEN 1 ELSE 0 END
            ) AS had_purchase

        FROM read_parquet('{parquet_path}')

        WHERE event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'

        GROUP BY
            month,
            user_id
    ),

    user_stage AS (
        SELECT
            month,
            user_id,

            CASE
                WHEN had_purchase = 1 THEN 'Purchase'
                WHEN had_cart = 1 THEN 'Cart but no Purchase'
                ELSE 'View only'
            END AS user_stage

        FROM monthly_user_behavior
    ),

    user_revisit AS (
        SELECT
            a.month,
            a.user_id,
            a.user_stage,

            CASE
                WHEN b.user_id IS NOT NULL THEN 1
                ELSE 0
            END AS revisited

        FROM user_stage a

        LEFT JOIN user_stage b
            ON a.user_id = b.user_id
           AND b.month = a.month + INTERVAL '1 month'

        WHERE a.month < DATE '2020-04-01'
    )

    SELECT
        STRFTIME(month, '%Y-%m') AS base_month,
        STRFTIME(month + INTERVAL '1 month', '%Y-%m') AS next_month,

        user_stage,

        COUNT(*) AS user_count,
        SUM(revisited) AS revisit_users,

        ROUND(
            SUM(revisited) * 100.0 / COUNT(*),
            2
        ) AS revisit_rate

    FROM user_revisit

    GROUP BY
        month,
        user_stage

    ORDER BY
        month,
        CASE user_stage
            WHEN 'View only' THEN 1
            WHEN 'Cart but no Purchase' THEN 2
            WHEN 'Purchase' THEN 3
        END
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────────────────┬────────────┬───────────────┬──────────────┐
│ base_month │ next_month │      user_stage      │ user_count │ revisit_users │ revisit_rate │
│  varchar   │  varchar   │       varchar        │   int64    │    int128     │    double    │
├────────────┼────────────┼──────────────────────┼────────────┼───────────────┼──────────────┤
│ 2019-12    │ 2020-01    │ View only            │    3649958 │       1282058 │        35.13 │
│ 2019-12    │ 2020-01    │ Cart but no Purchase │     426277 │        225372 │        52.87 │
│ 2019-12    │ 2020-01    │ Purchase             │     500997 │        289998 │        57.88 │
│ 2020-01    │ 2020-02    │ View only            │    3641845 │       1284564 │        35.27 │
│ 2020-01    │ 2020-02    │ Cart but no Purchase │     385035 │        212999 │        55.32 │
│ 2020-01    │ 2020-02    │ Purchase             │     359105 │        205160 │        57.13 │
│ 2020-02    │ 2020-03    │ View only            │

In [28]:
# 사용자 행동 단계별 규모와 다음 달 재방문 사용자 기여 비중 확인

duckdb.sql(f"""
    WITH monthly_user_behavior AS (
        SELECT
            DATE_TRUNC('month', event_time) AS month,
            user_id,

            MAX(
                CASE WHEN event_type = 'cart'
                     THEN 1 ELSE 0 END
            ) AS had_cart,

            MAX(
                CASE WHEN event_type = 'purchase'
                     THEN 1 ELSE 0 END
            ) AS had_purchase

        FROM read_parquet('{parquet_path}')

        WHERE event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'

        GROUP BY
            month,
            user_id
    ),

    user_stage AS (
        SELECT
            month,
            user_id,

            CASE
                WHEN had_purchase = 1 THEN 'Purchase'
                WHEN had_cart = 1 THEN 'Cart but no Purchase'
                ELSE 'View only'
            END AS user_stage

        FROM monthly_user_behavior
    ),

    user_revisit AS (
        SELECT
            a.month,
            a.user_id,
            a.user_stage,

            CASE
                WHEN b.user_id IS NOT NULL THEN 1
                ELSE 0
            END AS revisited

        FROM user_stage a

        LEFT JOIN user_stage b
            ON a.user_id = b.user_id
           AND b.month = a.month + INTERVAL '1 month'

        WHERE a.month < DATE '2020-04-01'
    ),

    stage_summary AS (
        SELECT
            month,
            user_stage,
            COUNT(*) AS user_count,
            SUM(revisited) AS revisit_users

        FROM user_revisit

        GROUP BY
            month,
            user_stage
    )

    SELECT
        STRFTIME(month, '%Y-%m') AS base_month,
        user_stage,
        user_count,

        ROUND(
            user_count * 100.0
            / SUM(user_count) OVER (PARTITION BY month),
            2
        ) AS user_share_pct,

        revisit_users,

        ROUND(
            revisit_users * 100.0 / user_count,
            2
        ) AS revisit_rate,

        ROUND(
            revisit_users * 100.0
            / SUM(revisit_users) OVER (PARTITION BY month),
            2
        ) AS revisit_user_share_pct

    FROM stage_summary

    ORDER BY
        month,
        CASE user_stage
            WHEN 'View only' THEN 1
            WHEN 'Cart but no Purchase' THEN 2
            WHEN 'Purchase' THEN 3
        END
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┬────────────┬────────────────┬───────────────┬──────────────┬────────────────────────┐
│ base_month │      user_stage      │ user_count │ user_share_pct │ revisit_users │ revisit_rate │ revisit_user_share_pct │
│  varchar   │       varchar        │   int64    │     double     │    int128     │    double    │         double         │
├────────────┼──────────────────────┼────────────┼────────────────┼───────────────┼──────────────┼────────────────────────┤
│ 2019-12    │ View only            │    3649958 │          79.74 │       1282058 │        35.13 │                  71.33 │
│ 2019-12    │ Cart but no Purchase │     426277 │           9.31 │        225372 │        52.87 │                  12.54 │
│ 2019-12    │ Purchase             │     500997 │          10.95 │        289998 │        57.88 │                  16.13 │
│ 2020-01    │ View only            │    3641845 │          83.03 │       1284564 │        35.27 │                  75.44 │
│ 2020-0

In [2]:
#--------------------------------------------------------------------------------------------------------------------------#

In [8]:
# [환경 설정]
# Tableau용 CSV 저장 폴더 설정

marts_dir = Path(r"../data/marts")
marts_dir.mkdir(parents=True, exist_ok=True)

In [10]:
# [Tableau용 저장 파일]
# 월→다음 달 전체 사용자 재방문율 저장

monthly_revisit_csv = r"../data/marts/dashboard_monthly_revisit.csv"

duckdb.sql(f"""
    COPY (
        WITH monthly_users AS (
            SELECT DISTINCT
                DATE_TRUNC('month', event_time) AS month,
                user_id

            FROM read_parquet('{parquet_path}')

            WHERE event_time >= '{analysis_start_date}'
              AND event_time < '{analysis_end_date}'
        ),

        revisit AS (
            SELECT
                a.month,
                COUNT(DISTINCT a.user_id) AS base_users,
                COUNT(DISTINCT b.user_id) AS revisit_users

            FROM monthly_users a

            LEFT JOIN monthly_users b
                ON a.user_id = b.user_id
               AND b.month = a.month + INTERVAL '1 month'

            WHERE a.month < DATE '2020-04-01'

            GROUP BY a.month
        )

        SELECT
            STRFTIME(month, '%Y-%m') AS base_month,
            STRFTIME(
                month + INTERVAL '1 month',
                '%Y-%m'
            ) AS next_month,

            base_users,
            revisit_users,

            ROUND(
                revisit_users * 100.0 / base_users,
                2
            ) AS revisit_rate

        FROM revisit

        ORDER BY month
    )
    TO '{monthly_revisit_csv}'
    (HEADER, DELIMITER ',')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
# [검증용 코드]
# 월별 재방문율 CSV 저장 결과 확인

duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{monthly_revisit_csv}')
""").show()

┌────────────┬────────────┬────────────┬───────────────┬──────────────┐
│ base_month │ next_month │ base_users │ revisit_users │ revisit_rate │
│  varchar   │  varchar   │   int64    │     int64     │    double    │
├────────────┼────────────┼────────────┼───────────────┼──────────────┤
│ 2019-12    │ 2020-01    │    4577232 │       1797428 │        39.27 │
│ 2020-01    │ 2020-02    │    4385985 │       1702723 │        38.82 │
│ 2020-02    │ 2020-03    │    4233206 │       1659585 │         39.2 │
│ 2020-03    │ 2020-04    │    4114060 │       1526245 │         37.1 │
└────────────┴────────────┴────────────┴───────────────┴──────────────┘



In [15]:
# [Tableau용 저장 파일]
# 사용자 행동 단계별 다음 달 재방문율과 사용자 구성비 저장

revisit_by_stage_csv = r"../data/marts/dashboard_revisit_by_stage.csv"

duckdb.sql(f"""
    COPY (
        WITH monthly_user_behavior AS (
            SELECT
                DATE_TRUNC('month', event_time) AS month,
                user_id,

                MAX(
                    CASE
                        WHEN event_type = 'cart' THEN 1
                        ELSE 0
                    END
                ) AS had_cart,

                MAX(
                    CASE
                        WHEN event_type = 'purchase' THEN 1
                        ELSE 0
                    END
                ) AS had_purchase

            FROM read_parquet('{parquet_path}')

            WHERE event_time >= '{analysis_start_date}'
              AND event_time < '{analysis_end_date}'

            GROUP BY
                month,
                user_id
        ),

        user_stage AS (
            SELECT
                month,
                user_id,

                CASE
                    WHEN had_purchase = 1
                        THEN 'Purchase'
                    WHEN had_cart = 1
                        THEN 'Cart but no Purchase'
                    ELSE 'View only'
                END AS user_stage

            FROM monthly_user_behavior
        ),

        user_revisit AS (
            SELECT
                a.month,
                a.user_id,
                a.user_stage,

                CASE
                    WHEN b.user_id IS NOT NULL THEN 1
                    ELSE 0
                END AS revisited

            FROM user_stage a

            LEFT JOIN user_stage b
                ON a.user_id = b.user_id
               AND b.month = a.month + INTERVAL '1 month'

            WHERE a.month < DATE '2020-04-01'
        ),

        stage_summary AS (
            SELECT
                month,
                user_stage,
                COUNT(*) AS user_count,
                SUM(revisited) AS revisit_users

            FROM user_revisit

            GROUP BY
                month,
                user_stage
        )

        SELECT
            STRFTIME(month, '%Y-%m') AS base_month,

            STRFTIME(
                month + INTERVAL '1 month',
                '%Y-%m'
            ) AS next_month,

            user_stage,
            user_count,

            ROUND(
                user_count * 100.0
                / SUM(user_count) OVER (
                    PARTITION BY month
                ),
                2
            ) AS user_share_pct,

            revisit_users,

            ROUND(
                revisit_users * 100.0 / user_count,
                2
            ) AS revisit_rate,

            ROUND(
                revisit_users * 100.0
                / SUM(revisit_users) OVER (
                    PARTITION BY month
                ),
                2
            ) AS revisit_user_share_pct

        FROM stage_summary

        ORDER BY
            month,
            CASE user_stage
                WHEN 'View only' THEN 1
                WHEN 'Cart but no Purchase' THEN 2
                WHEN 'Purchase' THEN 3
            END
    )
    TO '{revisit_by_stage_csv}'
    (HEADER, DELIMITER ',')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
# [검증용 코드]
# 행동 단계별 재방문율 CSV 저장 결과 확인

duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{revisit_by_stage_csv}')
""").show()

┌────────────┬────────────┬──────────────────────┬────────────┬────────────────┬───────────────┬──────────────┬────────────────────────┐
│ base_month │ next_month │      user_stage      │ user_count │ user_share_pct │ revisit_users │ revisit_rate │ revisit_user_share_pct │
│  varchar   │  varchar   │       varchar        │   int64    │     double     │     int64     │    double    │         double         │
├────────────┼────────────┼──────────────────────┼────────────┼────────────────┼───────────────┼──────────────┼────────────────────────┤
│ 2019-12    │ 2020-01    │ View only            │    3649958 │          79.74 │       1282058 │        35.13 │                  71.33 │
│ 2019-12    │ 2020-01    │ Cart but no Purchase │     426277 │           9.31 │        225372 │        52.87 │                  12.54 │
│ 2019-12    │ 2020-01    │ Purchase             │     500997 │          10.95 │        289998 │        57.88 │                  16.13 │
│ 2020-01    │ 2020-02    │ View only    